In [1]:
import numpy as np

# ---------------- Node Class ----------------
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature        # index of feature to split on
        self.threshold = threshold    # value to split at
        self.left = left              # left subtree
        self.right = right            # right subtree
        self.value = value            # prediction at leaf (if any)

# ---------------- Decision Tree Classifier ----------------
class DecisionTree:
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # Entry point for training
    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    # Recursively builds the tree
    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        num_labels = len(np.unique(y))

        # Stopping conditions
        if (depth >= self.max_depth or num_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Find best split
        best_feature, best_thresh = self._best_split(X, y)

        # Split the data
        left_idxs = X[:, best_feature] <= best_thresh
        right_idxs = X[:, best_feature] > best_thresh

        left = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)

        return Node(feature=best_feature, threshold=best_thresh, left=left, right=right)

    # Find the best feature and threshold to split
    def _best_split(self, X, y):
        best_gain = -1
        best_feat = None
        best_thresh = None

        for feature in range(X.shape[1]):
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                gain = self._information_gain(X, y, feature, threshold)

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feature
                    best_thresh = threshold

        return best_feat, best_thresh

    # Gini impurity
    def _gini(self, y):
        classes = np.unique(y)
        gini = 1.0
        for c in classes:
            p = np.sum(y == c) / len(y)
            gini -= p ** 2
        return gini

    # Info gain using Gini
    def _information_gain(self, X, y, feature, threshold):
        parent_gini = self._gini(y)

        left_idxs = X[:, feature] <= threshold
        right_idxs = X[:, feature] > threshold

        if len(y[left_idxs]) == 0 or len(y[right_idxs]) == 0:
            return 0

        n = len(y)
        n_left = len(y[left_idxs])
        n_right = len(y[right_idxs])

        child_gini = (n_left / n) * self._gini(y[left_idxs]) + (n_right / n) * self._gini(y[right_idxs])
        ig = parent_gini - child_gini
        return ig

    # Most common label at leaf
    def _most_common_label(self, y):
        counts = np.bincount(y)
        return np.argmax(counts)

    # Prediction for a single sample
    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

    # Predict for all samples
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load sample data (we’ll only use 2 classes for binary classification)
data = load_iris()
X = data.data[data.target != 2]
y = data.target[data.target != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train the tree
tree = DecisionTree(max_depth=3)
tree.fit(X_train, y_train)

# Predict
preds = tree.predict(X_test)

# Accuracy
accuracy = np.mean(preds == y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 100.00%
